# Hugging Face Transformers 完整指南

本 notebook 是第14章的補充材料，介紹如何使用 Hugging Face 生態系統來快速實現各種 NLP 任務。

**學習目標：**
- 了解 Hugging Face 生態系統
- 掌握載入和使用預訓練模型的方法
- 學習常見 NLP 任務的實現
- 理解模型微調的最佳實踐
- 對比從零實現與使用庫的優缺點

## 1. Hugging Face 生態系統簡介

Hugging Face 提供了一整套工具，讓我們能夠輕鬆使用預訓練模型。主要組件包括：

### 1.1 核心庫

1. **Transformers 庫**
   - 提供數千個預訓練模型
   - 支援 PyTorch、TensorFlow、JAX
   - 統一的 API 設計

2. **Datasets 庫**
   - 提供大量標準數據集
   - 高效的數據加載和處理
   - 內建數據預處理功能

3. **Tokenizers 庫**
   - 快速的分詞器實現（Rust 後端）
   - 支援各種分詞算法
   - 與 Transformers 無縫整合

4. **Hub 模型倉庫**
   - 社群共享的模型和數據集
   - 版本控制和協作功能
   - 簡單的模型部署

In [ ]:
# 安裝必要的庫
# !pip install transformers datasets tokenizers torch torchvision torchaudio
# !pip install accelerate evaluate scikit-learn

In [ ]:
# 導入基本庫
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# 檢查可用設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")
print(f"Transformers 版本: {__import__('transformers').__version__}")

## 2. 基礎使用

### 2.1 載入預訓練模型

Hugging Face 提供了 `Auto*` 類別，可以根據模型名稱自動選擇正確的模型架構。

In [ ]:
# 方法 1: 使用 AutoModel（通用模型載入）
model_name = 'bert-base-uncased'

print("載入 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("載入模型...")
model = AutoModel.from_pretrained(model_name)
model = model.to(device)

print(f"\n模型參數量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"模型配置: {model.config}")

In [ ]:
# 方法 2: 使用任務專用的模型類別
# 例如：文本分類任務
model_for_classification = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased-finetuned-sst-2-english'
)
model_for_classification = model_for_classification.to(device)

print(f"分類模型類別數: {model_for_classification.config.num_labels}")

### 2.2 Tokenization 深入理解

不同模型使用不同的分詞算法：
- **WordPiece**: BERT 使用，將詞分解為子詞單元
- **BPE** (Byte Pair Encoding): GPT 系列使用
- **SentencePiece**: T5、XLNet 等使用
- **Unigram**: XLM-RoBERTa 等使用

In [ ]:
# 2.2.1 WordPiece Tokenization (BERT)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "Hugging Face is democratizing NLP!"
print("原始文本:", text)
print("\n=== BERT (WordPiece) ===")

# 基本分詞
tokens = bert_tokenizer.tokenize(text)
print("Tokens:", tokens)

# 轉換為 ID
token_ids = bert_tokenizer.encode(text)
print("Token IDs:", token_ids)

# 完整編碼（包含特殊 tokens）
encoded = bert_tokenizer(text, return_tensors='pt')
print("\n完整編碼結果:")
print(f"  input_ids: {encoded['input_ids']}")
print(f"  attention_mask: {encoded['attention_mask']}")

# 解碼回文本
decoded = bert_tokenizer.decode(encoded['input_ids'][0])
print(f"\n解碼後: {decoded}")

In [ ]:
# 2.2.2 BPE Tokenization (GPT-2)
from transformers import GPT2Tokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

print("=== GPT-2 (BPE) ===")
tokens = gpt2_tokenizer.tokenize(text)
print("Tokens:", tokens)
print("Token IDs:", gpt2_tokenizer.encode(text))

# 注意：GPT-2 使用特殊的空格標記 'Ġ'
print("\n注意 'Ġ' 符號表示該 token 前有空格")

In [ ]:
# 2.2.3 處理長文本和批次數據
long_texts = [
    "This is a short sentence.",
    "This is a much longer sentence that needs to be truncated or padded.",
    "Short."
]

# 批次編碼，自動填充和截斷
batch_encoding = bert_tokenizer(
    long_texts,
    padding=True,              # 填充到批次中最長的序列
    truncation=True,           # 截斷超過最大長度的序列
    max_length=20,             # 最大序列長度
    return_tensors='pt'        # 返回 PyTorch tensors
)

print("批次編碼結果:")
print(f"input_ids shape: {batch_encoding['input_ids'].shape}")
print(f"\ninput_ids:\n{batch_encoding['input_ids']}")
print(f"\nattention_mask:\n{batch_encoding['attention_mask']}")

# 解碼每個句子
print("\n解碼後的文本:")
for i, ids in enumerate(batch_encoding['input_ids']):
    decoded = bert_tokenizer.decode(ids, skip_special_tokens=True)
    print(f"  {i+1}. {decoded}")

In [ ]:
# 2.2.4 特殊 tokens 處理
print("BERT 特殊 tokens:")
print(f"  [CLS] token: {bert_tokenizer.cls_token} (ID: {bert_tokenizer.cls_token_id})")
print(f"  [SEP] token: {bert_tokenizer.sep_token} (ID: {bert_tokenizer.sep_token_id})")
print(f"  [PAD] token: {bert_tokenizer.pad_token} (ID: {bert_tokenizer.pad_token_id})")
print(f"  [UNK] token: {bert_tokenizer.unk_token} (ID: {bert_tokenizer.unk_token_id})")
print(f"  [MASK] token: {bert_tokenizer.mask_token} (ID: {bert_tokenizer.mask_token_id})")

# 句子對編碼（用於 NSP 等任務）
sentence_a = "How are you?"
sentence_b = "I am fine, thank you!"

encoded_pair = bert_tokenizer(
    sentence_a, 
    sentence_b,
    return_tensors='pt',
    add_special_tokens=True
)

print(f"\n句子對編碼: {encoded_pair['input_ids']}")
decoded_pair = bert_tokenizer.decode(encoded_pair['input_ids'][0])
print(f"解碼: {decoded_pair}")

# token_type_ids 區分兩個句子
print(f"\ntoken_type_ids: {encoded_pair['token_type_ids']}")
print("  0 表示第一個句子，1 表示第二個句子")

### 2.3 簡單推論示例

In [ ]:
# 基本推論流程
text = "Hello, how are you?"

# 1. Tokenize
inputs = tokenizer(text, return_tensors='pt').to(device)

# 2. 前向傳播
with torch.no_grad():
    outputs = model(**inputs)

# 3. 獲取輸出
last_hidden_states = outputs.last_hidden_state
pooler_output = outputs.pooler_output  # [CLS] token 的表示

print(f"輸入文本: {text}")
print(f"\nLast hidden states shape: {last_hidden_states.shape}")
print(f"  (batch_size, sequence_length, hidden_size)")
print(f"\nPooler output shape: {pooler_output.shape}")
print(f"  (batch_size, hidden_size)")

# 4. 使用句子嵌入（可用於相似度計算等）
sentence_embedding = pooler_output.cpu().numpy()
print(f"\n句子嵌入前5個維度: {sentence_embedding[0, :5]}")

## 3. 常見任務

### 3.1 文本分類

#### 3.1.1 使用 Pipeline（最簡單的方法）

In [ ]:
# 創建情感分析 pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

# 單個文本
result = classifier("I love this product! It's amazing!")
print("單個文本結果:", result)

# 批次處理
texts = [
    "I love this product!",
    "This is terrible.",
    "Not sure how I feel about this.",
    "Best purchase ever!"
]

results = classifier(texts)
print("\n批次處理結果:")
for text, result in zip(texts, results):
    print(f"  {text:40s} -> {result['label']:8s} ({result['score']:.4f})")

#### 3.1.2 自定義訓練（使用小數據集演示）

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding
import evaluate

# 創建小型示例數據集
train_data = {
    'text': [
        "I love this movie", "Great product", "Excellent service",
        "Terrible experience", "Very disappointed", "Waste of money",
        "Not bad", "Could be better", "It's okay",
        "Absolutely fantastic", "Highly recommend", "Amazing quality"
    ],
    'label': [1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1]  # 1: positive, 0: negative
}

test_data = {
    'text': [
        "I really enjoyed it",
        "This is awful",
        "Pretty good",
        "Not recommended"
    ],
    'label': [1, 0, 1, 0]
}

train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

print(f"訓練集大小: {len(train_dataset)}")
print(f"測試集大小: {len(test_dataset)}")
print(f"\n示例數據: {train_dataset[0]}")

In [ ]:
# 準備數據
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

# Tokenize 數據集
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 創建 data collator（用於動態填充）
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("數據預處理完成！")
print(f"Tokenized 示例: {tokenized_train[0]}")

In [ ]:
# 載入模型並設置訓練參數
from transformers import Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 定義評估指標
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 設置訓練參數
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 創建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer 已準備就緒！")

In [ ]:
# 開始訓練（這可能需要幾分鐘）
print("開始訓練...")
trainer.train()
print("\n訓練完成！")

In [ ]:
# 評估模型
results = trainer.evaluate()
print("\n評估結果:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

# 進行預測
test_texts = [
    "This is wonderful!",
    "I hate this.",
    "It's fantastic!"
]

# Tokenize 測試文本
test_encodings = tokenizer(test_texts, truncation=True, padding=True, return_tensors='pt')
test_encodings = {k: v.to(model.device) for k, v in test_encodings.items()}

# 預測
model.eval()
with torch.no_grad():
    outputs = model(**test_encodings)
    predictions = torch.argmax(outputs.logits, dim=-1)

label_names = ['negative', 'positive']
print("\n預測結果:")
for text, pred in zip(test_texts, predictions):
    print(f"  {text:25s} -> {label_names[pred]}")

### 3.2 命名實體識別 (NER)

In [ ]:
# 使用 Pipeline 進行 NER
ner_pipeline = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",  # 合併同一實體的多個 tokens
    device=0 if torch.cuda.is_available() else -1
)

text = """Apple Inc. was founded by Steve Jobs in Cupertino, California. 
The company is now led by Tim Cook and has offices in New York and London."""

entities = ner_pipeline(text)

print("命名實體識別結果:\n")
for entity in entities:
    print(f"實體: {entity['word']:20s} | 類型: {entity['entity_group']:10s} | 信心度: {entity['score']:.4f}")

# 視覺化標註
print("\n\n原文標註:")
print(text)
print("\n標註說明:")
print("  PER: 人名 (Person)")
print("  ORG: 組織 (Organization)")
print("  LOC: 地點 (Location)")

In [ ]:
# 手動進行 NER（更細粒度的控制）
from transformers import AutoModelForTokenClassification

model_name = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model = model.to(device)

text = "Hugging Face is based in New York City."
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)

# 獲取標籤名稱
labels = [model.config.id2label[pred.item()] for pred in predictions[0]]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Token 級別的 NER 結果:\n")
for token, label in zip(tokens, labels):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"{token:20s} -> {label}")

### 3.3 問答系統

In [ ]:
# 使用 Pipeline 進行問答
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=0 if torch.cuda.is_available() else -1
)

context = """
Transformers is a library developed by Hugging Face that provides thousands of 
pretrained models for natural language processing tasks. The library supports 
PyTorch, TensorFlow, and JAX frameworks. It was first released in 2018 and has 
become one of the most popular NLP libraries in the world.
"""

questions = [
    "Who developed Transformers?",
    "When was it released?",
    "What frameworks does it support?",
    "What does the library provide?"
]

print("問答系統演示\n")
print(f"上下文: {context.strip()}\n")
print("=" * 80)

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"\n問題: {question}")
    print(f"答案: {result['answer']}")
    print(f"信心度: {result['score']:.4f}")
    print(f"位置: [{result['start']}:{result['end']}]")

In [ ]:
# 手動實現問答（理解內部機制）
from transformers import AutoModelForQuestionAnswering

model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model = model.to(device)

question = "What is Transformers?"
context = "Transformers is a library for natural language processing."

# 編碼問題和上下文
inputs = tokenizer(question, context, return_tensors="pt").to(device)

# 前向傳播
with torch.no_grad():
    outputs = model(**inputs)

# 獲取答案的起始和結束位置
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

# 解碼答案
answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end])
)

print(f"問題: {question}")
print(f"上下文: {context}")
print(f"\n預測答案: {answer}")
print(f"起始位置信心度: {torch.softmax(outputs.start_logits, dim=1)[0][answer_start]:.4f}")
print(f"結束位置信心度: {torch.softmax(outputs.end_logits, dim=1)[0][answer_end-1]:.4f}")

### 3.4 文本生成

In [ ]:
# 使用 GPT-2 進行文本生成
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model = model.to(device)

# 設置 pad_token（GPT-2 預設沒有）
tokenizer.pad_token = tokenizer.eos_token

prompt = "Once upon a time, in a land far away,"

print(f"提示詞: {prompt}\n")
print("=" * 80)

# 編碼提示詞
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# 生成文本（貪婪解碼）
print("\n1. 貪婪解碼 (Greedy Decoding):")
with torch.no_grad():
    greedy_output = model.generate(
        input_ids,
        max_length=50,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

# Beam Search
print("\n2. Beam Search (num_beams=5):")
with torch.no_grad():
    beam_output = model.generate(
        input_ids,
        max_length=50,
        num_beams=5,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        early_stopping=True
    )
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

# Top-k Sampling
print("\n3. Top-k Sampling (k=50):")
with torch.no_grad():
    topk_output = model.generate(
        input_ids,
        max_length=50,
        do_sample=True,
        top_k=50,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(topk_output[0], skip_special_tokens=True))

# Top-p (Nucleus) Sampling
print("\n4. Top-p Sampling (p=0.92):")
with torch.no_grad():
    topp_output = model.generate(
        input_ids,
        max_length=50,
        do_sample=True,
        top_p=0.92,
        top_k=0,  # 禁用 top-k
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(topp_output[0], skip_special_tokens=True))

# 溫度採樣
print("\n5. 溫度採樣 (temperature=0.7):")
with torch.no_grad():
    temp_output = model.generate(
        input_ids,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(temp_output[0], skip_special_tokens=True))

In [ ]:
# 生成多個不同的續寫
print("生成多個不同版本:\n")

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_length=40,
        num_return_sequences=3,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

for i, output in enumerate(outputs):
    text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"版本 {i+1}:")
    print(text)
    print()

## 4. 模型微調

### 4.1 使用 Trainer API（推薦）

我們已經在 3.1.2 中演示了使用 Trainer API 進行微調。這裡展示一些高級配置。

In [ ]:
# 高級 TrainingArguments 配置
from transformers import TrainingArguments

advanced_training_args = TrainingArguments(
    # 基本設置
    output_dir="./advanced_results",
    overwrite_output_dir=True,
    
    # 訓練參數
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    # 優化器設置
    learning_rate=2e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    
    # 學習率調度
    lr_scheduler_type="linear",
    warmup_steps=500,
    # warmup_ratio=0.1,  # 也可以使用比例
    
    # 評估策略
    eval_strategy="steps",
    eval_steps=100,
    
    # 保存策略
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,  # 只保留最近的 3 個檢查點
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # 日誌記錄
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    
    # 混合精度訓練
    fp16=torch.cuda.is_available(),  # 使用 FP16（需要 GPU）
    
    # 梯度累積
    gradient_accumulation_steps=2,  # 有效批次大小 = 8 * 2 = 16
    
    # 其他
    seed=42,
    report_to=["tensorboard"],  # 報告到 TensorBoard
    push_to_hub=False,  # 是否推送到 Hugging Face Hub
)

print("進階訓練配置:")
print(f"  有效批次大小: {advanced_training_args.per_device_train_batch_size * advanced_training_args.gradient_accumulation_steps}")
print(f"  混合精度: {advanced_training_args.fp16}")
print(f"  學習率調度: {advanced_training_args.lr_scheduler_type}")
print(f"  梯度累積步數: {advanced_training_args.gradient_accumulation_steps}")

In [ ]:
# 自定義評估指標
import evaluate

# 載入多個評估指標
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics_advanced(eval_pred):
    """計算多個評估指標"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return {
        'accuracy': accuracy.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1.compute(predictions=predictions, references=labels, average='weighted')['f1'],
        'precision': precision.compute(predictions=predictions, references=labels, average='weighted')['precision'],
        'recall': recall.compute(predictions=predictions, references=labels, average='weighted')['recall'],
    }

print("自定義評估指標已設置")
print("將計算: Accuracy, F1, Precision, Recall")

In [ ]:
# 自定義 Trainer（添加額外功能）
from transformers import Trainer

class CustomTrainer(Trainer):
    """自定義 Trainer 類別，添加額外的功能"""
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        可以自定義損失函數
        例如：添加正則化項、使用 label smoothing 等
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # 標準交叉熵損失
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        # 可以在這裡添加額外的損失項
        # 例如：L2 正則化
        # l2_reg = torch.tensor(0., device=loss.device)
        # for param in model.parameters():
        #     l2_reg += torch.norm(param)
        # loss += 0.01 * l2_reg
        
        return (loss, outputs) if return_outputs else loss
    
    def log(self, logs):
        """自定義日誌記錄"""
        if self.state.epoch is not None:
            logs["epoch"] = round(self.state.epoch, 2)
        
        # 可以添加自定義日誌
        if "loss" in logs:
            logs["custom_metric"] = logs["loss"] * 0.5  # 示例
        
        super().log(logs)

print("CustomTrainer 類別已定義")
print("  - 可自定義損失函數")
print("  - 可添加額外的日誌記錄")

### 4.2 自定義訓練循環

有時我們需要更細粒度的控制，這時可以使用 PyTorch 的原生訓練循環。

In [ ]:
# 自定義訓練循環示例
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

def custom_train_loop(model, train_dataset, eval_dataset, epochs=3):
    """完全自定義的訓練循環"""
    
    # 準備數據加載器
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        collate_fn=data_collator
    )
    
    eval_dataloader = DataLoader(
        eval_dataset,
        batch_size=8,
        collate_fn=data_collator
    )
    
    # 設置優化器
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    # 計算訓練步數
    num_training_steps = epochs * len(train_dataloader)
    num_warmup_steps = num_training_steps // 10
    
    # 學習率調度器
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # 訓練循環
    model.train()
    global_step = 0
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        epoch_loss = 0
        
        progress_bar = tqdm(train_dataloader, desc="Training")
        for batch in progress_bar:
            # 將數據移到設備
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # 前向傳播
            outputs = model(**batch)
            loss = outputs.loss
            
            # 反向傳播
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 更新參數
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            epoch_loss += loss.item()
            global_step += 1
            
            # 更新進度條
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })
        
        avg_train_loss = epoch_loss / len(train_dataloader)
        print(f"Average training loss: {avg_train_loss:.4f}")
        
        # 評估
        model.eval()
        eval_loss = 0
        predictions_list = []
        labels_list = []
        
        with torch.no_grad():
            for batch in tqdm(eval_dataloader, desc="Evaluating"):
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                
                eval_loss += outputs.loss.item()
                predictions = torch.argmax(outputs.logits, dim=-1)
                
                predictions_list.extend(predictions.cpu().numpy())
                labels_list.extend(batch['labels'].cpu().numpy())
        
        avg_eval_loss = eval_loss / len(eval_dataloader)
        eval_accuracy = accuracy.compute(
            predictions=predictions_list,
            references=labels_list
        )['accuracy']
        
        print(f"Evaluation loss: {avg_eval_loss:.4f}")
        print(f"Evaluation accuracy: {eval_accuracy:.4f}")
        
        model.train()
    
    return model

print("自定義訓練循環函數已定義")
print("\n特點:")
print("  - 完全控制訓練過程")
print("  - 可以添加任何自定義邏輯")
print("  - 包含梯度裁剪")
print("  - 學習率 warmup")
print("  - 訓練和評估循環")

In [ ]:
# 使用自定義訓練循環（可選執行）
# 注意：這會重新訓練模型

# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased",
#     num_labels=2
# ).to(device)

# trained_model = custom_train_loop(
#     model,
#     tokenized_train,
#     tokenized_test,
#     epochs=2
# )

print("若要使用自定義訓練循環，請取消註釋上方代碼")

## 5. 實用技巧

### 5.1 模型並行與數據並行

In [ ]:
# 數據並行（多 GPU 訓練）
print("數據並行示例\n")

if torch.cuda.device_count() > 1:
    print(f"檢測到 {torch.cuda.device_count()} 個 GPU")
    
    # 方法 1: 使用 PyTorch 的 DataParallel
    model = AutoModel.from_pretrained('bert-base-uncased')
    model = torch.nn.DataParallel(model)
    model = model.to(device)
    print("使用 DataParallel 包裝模型")
    
    # 方法 2: 使用 Trainer 的內建支持
    # TrainingArguments 會自動檢測多 GPU 並使用 DistributedDataParallel
    training_args = TrainingArguments(
        output_dir="./multi_gpu_results",
        per_device_train_batch_size=8,
        # Trainer 會自動處理多 GPU
    )
    print("Trainer 會自動使用所有可用的 GPU")
else:
    print("只有一個 GPU 或使用 CPU")
    print("\n數據並行配置（當有多個 GPU 時）:")
    print("""
    # 使用 PyTorch DataParallel
    model = torch.nn.DataParallel(model)
    
    # 或使用 Trainer，它會自動檢測並使用多 GPU
    training_args = TrainingArguments(
        output_dir="./results",
        per_device_train_batch_size=8,
        # 自動使用所有可用 GPU
    )
    """)

In [ ]:
# 模型並行（大模型拆分到多個 GPU）
print("模型並行示例\n")
print("用於非常大的模型（如 GPT-3、BLOOM 等）")
print("""
# 使用 Accelerate 庫進行自動模型並行
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

# 在空權重下初始化模型（不佔用內存）
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained('bigscience/bloom-7b1')

# 自動將模型分配到多個 GPU
model = load_checkpoint_and_dispatch(
    model,
    checkpoint='bigscience/bloom-7b1',
    device_map='auto',  # 自動分配
    no_split_module_classes=['BloomBlock']  # 不拆分的模組
)
""")

print("\n手動指定設備映射:")
print("""
device_map = {
    'transformer.word_embeddings': 0,
    'transformer.h.0': 0,
    'transformer.h.1': 0,
    'transformer.h.2': 1,
    'transformer.h.3': 1,
    # ...
}
model = load_checkpoint_and_dispatch(model, checkpoint=..., device_map=device_map)
""")

### 5.2 混合精度訓練

In [ ]:
# 混合精度訓練示例
print("混合精度訓練 (FP16)\n")
print("優點:")
print("  - 減少內存使用（約 50%）")
print("  - 加快訓練速度（約 2-3x）")
print("  - 通常不會損失模型精度\n")

# 方法 1: 使用 Trainer（推薦）
training_args_fp16 = TrainingArguments(
    output_dir="./fp16_results",
    fp16=True,  # 啟用混合精度
    per_device_train_batch_size=16,  # 可以使用更大的批次
)

print("使用 Trainer 的 FP16 配置:")
print(f"  fp16={training_args_fp16.fp16}")

# 方法 2: 使用 PyTorch 原生支持
print("\n使用 PyTorch AMP (Automatic Mixed Precision):")
print("""
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for batch in dataloader:
    optimizer.zero_grad()
    
    # 使用 autocast 進行混合精度前向傳播
    with autocast():
        outputs = model(**batch)
        loss = outputs.loss
    
    # 縮放損失並反向傳播
    scaler.scale(loss).backward()
    
    # 更新權重
    scaler.step(optimizer)
    scaler.update()
""")

### 5.3 梯度累積

In [ ]:
# 梯度累積示例
print("梯度累積\n")
print("用途: 在 GPU 內存有限時模擬大批次訓練\n")

# 使用 Trainer
training_args_grad_accum = TrainingArguments(
    output_dir="./grad_accum_results",
    per_device_train_batch_size=4,      # 物理批次大小
    gradient_accumulation_steps=4,       # 累積步數
    # 有效批次大小 = 4 * 4 = 16
)

print("Trainer 配置:")
print(f"  物理批次大小: {training_args_grad_accum.per_device_train_batch_size}")
print(f"  梯度累積步數: {training_args_grad_accum.gradient_accumulation_steps}")
print(f"  有效批次大小: {training_args_grad_accum.per_device_train_batch_size * training_args_grad_accum.gradient_accumulation_steps}")

# 手動實現
print("\n手動實現梯度累積:")
print("""
accumulation_steps = 4
optimizer.zero_grad()

for i, batch in enumerate(dataloader):
    outputs = model(**batch)
    loss = outputs.loss
    
    # 將損失除以累積步數
    loss = loss / accumulation_steps
    loss.backward()
    
    # 每 accumulation_steps 步更新一次
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
""")

### 5.4 學習率調度

In [ ]:
# 學習率調度策略
import matplotlib.pyplot as plt

print("常見學習率調度策略\n")

# 模擬訓練步數
num_training_steps = 1000
num_warmup_steps = 100
initial_lr = 2e-5

# 創建假的優化器（用於演示）
dummy_model = torch.nn.Linear(10, 2)
optimizer = AdamW(dummy_model.parameters(), lr=initial_lr)

# 不同的調度器
from transformers import (
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
    get_constant_schedule_with_warmup,
)

schedulers = {
    'Linear': get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps, num_training_steps
    ),
    'Cosine': get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps, num_training_steps
    ),
    'Constant': get_constant_schedule_with_warmup(
        optimizer, num_warmup_steps
    ),
}

# 收集學習率
plt.figure(figsize=(12, 6))

for name, scheduler in schedulers.items():
    # 重置優化器
    for param_group in optimizer.param_groups:
        param_group['lr'] = initial_lr
    
    lrs = []
    for step in range(num_training_steps):
        lrs.append(scheduler.get_last_lr()[0])
        scheduler.step()
    
    plt.plot(lrs, label=name, linewidth=2)

plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Schedules', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.axvline(x=num_warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
plt.tight_layout()
plt.show()

print("\n在 Trainer 中使用:")
print("""
training_args = TrainingArguments(
    lr_scheduler_type='linear',  # 'linear', 'cosine', 'constant', etc.
    warmup_steps=500,
    # 或使用比例
    # warmup_ratio=0.1,
)
""")

### 5.5 保存和載入模型

In [ ]:
# 保存和載入模型
print("模型保存和載入\n")

# 創建一個示例模型
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# 保存模型和 tokenizer
save_directory = "./my_model"

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"模型已保存到: {save_directory}")

# 查看保存的文件
import os
print("\n保存的文件:")
for file in os.listdir(save_directory):
    print(f"  - {file}")

# 載入模型
print("\n載入模型...")
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)
print("模型載入成功！")

# 驗證載入的模型
test_text = "This is a test."
inputs = loaded_tokenizer(test_text, return_tensors="pt")
with torch.no_grad():
    outputs = loaded_model(**inputs)
print(f"\n測試預測: {torch.argmax(outputs.logits, dim=-1).item()}")

In [ ]:
# 推送模型到 Hugging Face Hub（可選）
print("推送模型到 Hugging Face Hub\n")
print("步驟:")
print("""
1. 登入 Hugging Face
   from huggingface_hub import login
   login()

2. 推送模型
   model.push_to_hub('my-username/my-model-name')
   tokenizer.push_to_hub('my-username/my-model-name')

3. 從 Hub 載入
   model = AutoModelForSequenceClassification.from_pretrained('my-username/my-model-name')
   tokenizer = AutoTokenizer.from_pretrained('my-username/my-model-name')
""")

print("\n使用 Trainer 自動推送:")
print("""
training_args = TrainingArguments(
    output_dir='./results',
    push_to_hub=True,
    hub_model_id='my-username/my-model-name',
    hub_strategy='every_save',  # 或 'end', 'checkpoint'
)
""")

## 6. 與本書內容的對比

### 6.1 從零實現 vs 使用庫

In [ ]:
# 比較表格
import pandas as pd

comparison_data = {
    '方面': [
        '學習深度',
        '開發速度',
        '代碼量',
        '靈活性',
        '性能優化',
        '維護成本',
        '生產就緒',
        '社群支持',
        '適用場景'
    ],
    '從零實現 (本書方法)': [
        '深入理解原理和細節',
        '較慢（需要實現所有細節）',
        '大量代碼',
        '完全控制每個細節',
        '需要手動優化',
        '高（需要自己修復 bug）',
        '需要大量額外工作',
        '有限（除非開源）',
        '學習、研究、實驗'
    ],
    '使用 Hugging Face': [
        '快速上手，深入需要額外學習',
        '非常快（幾行代碼即可）',
        '最小化代碼',
        '在框架內靈活',
        '內建優化（FP16、並行等）',
        '低（社群維護）',
        '開箱即用',
        '強大（大型社群）',
        '生產、快速原型、實際應用'
    ]
}

df = pd.DataFrame(comparison_data)
print("從零實現 vs 使用 Hugging Face 庫\n")
print(df.to_string(index=False))

print("\n\n推薦學習路徑:")
print("""
1. 階段一：從零實現（本書第14章）
   - 理解 Word2Vec、GloVe 的原理
   - 實現 BERT 的核心組件
   - 掌握預訓練的機制
   - 建立深厚的理論基礎

2. 階段二：學習 Hugging Face（本 notebook）
   - 了解如何使用預訓練模型
   - 掌握常見 NLP 任務的實現
   - 學習生產級最佳實踐
   - 提高開發效率

3. 階段三：深入理解
   - 閱讀 Transformers 源代碼
   - 理解庫的內部實現
   - 貢獻開源項目
   - 根據需求自定義組件
""")

### 6.2 實際代碼對比示例

In [ ]:
print("示例：情感分類任務\n")
print("=" * 80)

print("\n【從零實現（簡化版）】")
print("""
# 需要實現的組件:
1. 數據預處理
   - 分詞器
   - 詞彙表構建
   - 序列填充/截斷

2. 模型架構
   - Embedding 層
   - Multi-head Attention
   - Feed-forward 網絡
   - Layer Normalization
   - 位置編碼
   - 分類頭

3. 訓練流程
   - 數據加載器
   - 損失函數
   - 優化器
   - 訓練循環
   - 評估指標
   - 檢查點保存

總代碼量: 500-1000+ 行
開發時間: 數天到數週
""")

print("\n" + "=" * 80)
print("\n【使用 Hugging Face】")
print("""
from transformers import pipeline

# 使用預訓練模型（3 行代碼）
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

result = classifier("I love this!")
print(result)  # [{'label': 'POSITIVE', 'score': 0.9998}]

# 或者微調自己的模型（約 30 行代碼）
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased')
training_args = TrainingArguments(output_dir='./results', num_train_epochs=3)
trainer = Trainer(model=model, args=training_args, train_dataset=train_data)
trainer.train()

總代碼量: 10-50 行
開發時間: 數分鐘到數小時
""")

print("\n" + "=" * 80)

### 6.3 何時該用哪種方法

In [ ]:
print("使用建議\n")
print("=" * 80)

print("\n✅ 選擇從零實現（本書方法）當你：\n")
scenarios_from_scratch = [
    "想深入理解 Transformer 的工作原理",
    "需要實現全新的架構或創新",
    "進行學術研究，需要完全控制",
    "學習 NLP 和深度學習的基礎",
    "準備面試或考試",
    "需要理解每個組件的數學原理",
    "想要發表論文並提出新方法"
]
for i, scenario in enumerate(scenarios_from_scratch, 1):
    print(f"  {i}. {scenario}")

print("\n" + "=" * 80)
print("\n✅ 選擇 Hugging Face 當你：\n")
scenarios_hf = [
    "需要快速構建生產級應用",
    "想使用最先進的預訓練模型",
    "處理實際業務問題",
    "團隊協作開發項目",
    "需要可維護和可擴展的代碼",
    "想利用社群的最佳實踐",
    "時間和資源有限",
    "需要支持多種模型和任務"
]
for i, scenario in enumerate(scenarios_hf, 1):
    print(f"  {i}. {scenario}")

print("\n" + "=" * 80)
print("\n💡 最佳實踐：兩者結合\n")
print("""
理想的學習和工作流程:

1. 學習階段：從零實現
   - 徹底理解原理
   - 建立深厚基礎
   - 培養解決問題的能力

2. 開發階段：使用 Hugging Face
   - 快速原型開發
   - 利用預訓練模型
   - 遵循最佳實踐

3. 深入階段：理解庫的實現
   - 閱讀 Transformers 源代碼
   - 了解優化技巧
   - 必要時自定義組件

4. 貢獻階段：回饋社群
   - 修復 bug
   - 添加新功能
   - 分享經驗
""")

## 總結

本 notebook 介紹了 Hugging Face Transformers 生態系統的核心功能：

### 主要收穫

1. **生態系統理解**
   - Transformers、Datasets、Tokenizers 庫的作用
   - Hub 模型倉庫的使用

2. **基礎技能**
   - 載入和使用預訓練模型
   - 理解不同的分詞算法
   - 進行模型推論

3. **實際任務**
   - 文本分類、NER、問答、文本生成
   - 使用 Pipeline 快速實現
   - 自定義訓練流程

4. **進階技巧**
   - 模型並行和數據並行
   - 混合精度訓練
   - 梯度累積
   - 學習率調度

5. **實踐智慧**
   - 何時從零實現，何時使用庫
   - 如何結合兩種方法
   - 推薦的學習路徑

### 下一步學習

1. 嘗試其他預訓練模型（RoBERTa、ALBERT、T5 等）
2. 在自己的數據集上微調模型
3. 探索更多 NLP 任務（摘要、翻譯等）
4. 學習 Accelerate 庫進行大規模訓練
5. 閱讀 Transformers 源代碼深入理解

### 相關資源

- [Hugging Face 官方文檔](https://huggingface.co/docs/transformers)
- [Hugging Face Course](https://huggingface.co/course)
- [Transformers GitHub](https://github.com/huggingface/transformers)
- [Model Hub](https://huggingface.co/models)
- [Datasets Hub](https://huggingface.co/datasets)